In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [4]:
import pandas as pd
from collections import Counter, defaultdict
import random

train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

features = ["Pclass", "Sex", "Age", "Fare"]

X = train[features].copy()
y = train["Survived"]

X_test = test[features].copy()

X["Sex"] = X["Sex"].map({"male": 0, "female": 1})
X_test["Sex"] = X_test["Sex"].map({"male": 0, "female": 1})

X["Age"] = X["Age"].fillna(X["Age"].median())
X_test["Age"] = X_test["Age"].fillna(X["Age"].median())

X["Fare"] = X["Fare"].fillna(X["Fare"].median())
X_test["Fare"] = X_test["Fare"].fillna(X["Fare"].median())


def classification_error(y_true, y_pred):
    return sum(a != p for a, p in zip(y_true, y_pred)) / len(y_true)


In [5]:
class DecisionStumpClassifier:

    def __init__(self, feature_index=0):
        self.feature_index = feature_index

    def fit(self, X, y):
        groups = defaultdict(list)

        for x, label in zip(X, y):
            groups[x[self.feature_index]].append(label)

        self.rules = {
            value: Counter(labels).most_common(1)[0][0]
            for value, labels in groups.items()
        }

        self.default = Counter(y).most_common(1)[0][0]
        return self

    def predict(self, X):
        return [
            self.rules.get(x[self.feature_index], self.default)
            for x in X
        ]

In [6]:
class MemorizerClassifier:

    def fit(self, X, y):
        self.memory = dict(zip(map(tuple, X), y))
        self.labels = list(set(y))
        return self

    def predict(self, X):
        return [
            self.memory.get(tuple(x), random.choice(self.labels))
            for x in X
        ]

In [7]:
class MajorityVoteClassifier:

    def fit(self, X, y):
        self.label = Counter(y).most_common(1)[0][0]
        return self

    def predict(self, X):
        return [self.label] * len(X)

In [8]:
majority = MajorityVoteClassifier()
majority.fit(X.values.tolist(), y.tolist())
pred = majority.predict(X.values.tolist())

print("Majority Vote Accuracy:",
      1 - classification_error(y.tolist(), pred))


memorizer = MemorizerClassifier()
memorizer.fit(X.values.tolist(), y.tolist())
pred = memorizer.predict(X.values.tolist())

print("Memorizer Training Accuracy:",
      1 - classification_error(y.tolist(), pred))


best_stump = None
best_accuracy = 0

for i, feature in enumerate(features):
    stump = DecisionStumpClassifier(feature_index=i)
    stump.fit(X.values.tolist(), y.tolist())
    pred = stump.predict(X.values.tolist())

    accuracy = 1 - classification_error(y.tolist(), pred)

    print(feature, "Accuracy:", accuracy)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_stump = stump


test_predictions = best_stump.predict(X_test.values.tolist())

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print(submission.shape)

Majority Vote Accuracy: 0.6161616161616161
Memorizer Training Accuracy: 0.9607182940516273
Pclass Accuracy: 0.6790123456790124
Sex Accuracy: 0.7867564534231201
Age Accuracy: 0.6689113355780023
Fare Accuracy: 0.8035914702581369
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
(418, 2)
